# 03 · Router and Handoff

Pattern 1: a **router** looks at each customer message and **hands it over** to the right specialist.

```
Customer message ──> Router ──> Orders agent
                           ├──> Policy agent
                           └──> Human (when the agents should not handle it)
```

The router does not answer anything itself. It only **chooses**. For that we use **structured output** (Module 02): the router must return one of a fixed list of choices, so our code can act on it.

In [1]:
import json
from shop import client, MODEL     # the OpenAI client and the model name (see shop.py)

print("Ready")

Ready


In [2]:
from shop import run_agent, orders_agent, policy_agent     # from shop.py (built in notebook 02)

## The router
`Literal["orders", "policy", "human"]` means the model can only answer with one of those three words. It also gives a short `reason`, which helps us understand its choice.

In [3]:
from typing import Literal
from pydantic import BaseModel

class Route(BaseModel):
    specialist: Literal["orders", "policy", "human"]
    reason: str      # one short sentence

def choose_specialist(message):
    response = client.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "You route customer messages for an online shop. Choose one:\n"
                "- orders: questions about a specific order (status, delivery, details)\n"
                "- policy: questions about how the shop works (shipping, returns, warranty, payments)\n"
                "- human: angry customers, legal threats, or anything else the team cannot answer"
            )},
            {"role": "user", "content": message},
        ],
        response_format=Route,
    )
    return response.choices[0].message.parsed

## Handing over
Plain Python reads the router's choice and sends the message to the right agent. A dictionary maps each choice to an agent.

In [4]:
specialists = {
    "orders": orders_agent,
    "policy": policy_agent,
}

def handle(message):
    print("Customer:", message)

    route = choose_specialist(message)
    print("  [Router] ->", route.specialist, "(" + route.reason + ")")

    if route.specialist == "human":
        print("  Passed to a human agent.\n")
        return

    agent = specialists[route.specialist]
    answer = run_agent(agent, message)
    print("  Reply:", answer, "\n")

## Try different messages

In [5]:
messages = [
    "Where is my order A1002?",
    "Can I change my delivery address after ordering?",
    "This is outrageous. I will contact my lawyer if nobody calls me today!",
]

for message in messages:
    handle(message)

Customer: Where is my order A1002?


  [Router] -> orders (Customer is inquiring about the status of their specific order.)


  [Orders agent] lookup_order({'order_id': 'A1002'})


  Reply: Your order A1002 for running shoes is currently in transit and is expected to be delivered on November 25, 2026. 

Customer: Can I change my delivery address after ordering?


  [Router] -> policy (The question is about how the shop handles changes to delivery addresses after an order has been placed.)


  [Policy agent] search_policies({'question': 'changing delivery address after ordering'})


  Reply: Yes, you can change the delivery address until the order ships. Just email support with your order number and the new address. 

Customer: This is outrageous. I will contact my lawyer if nobody calls me today!


  [Router] -> human (Customer is expressing outrage and threatening legal action.)
  Passed to a human agent.



Each message went to the right place, and the angry one went to a **human**. A good system knows when **not** to let an agent answer.

## Recap
* A **router** picks the specialist. It uses structured output so its choice is one of a fixed list.
* Plain Python then **hands the message over**. There is no magic, just a dictionary and an `if`.
* Always include a **human** route for what the agents should not handle.

> ⚠️ The router can be wrong. If it sends a policy question to the Orders agent, that agent will say it cannot help (notebook 02). Log every routing decision so you can spot mistakes.

Next: what if a single question needs **two** specialists?